# Post 3 — Kaggle submission walkthrough

Step-by-step build of a Kaggle `submission.csv` for the Favorita Store Sales competition using the `src/demand` pipeline (LightGBM Tweedie on the `full` engineered feature set).

Each cell is a standalone step so you can stop, inspect, and re-run from any point. The end product is a populated copy of `sample_submission.csv` written to `results/sanity_check/submission.csv`.

Pipeline mirrors `src/demand/sanity_check/kaggle_pipeline.py` (PRD §11) — keep that file in sync if you change something here.

## 0. Imports, config, paths

In [ ]:
import json
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

from src.demand.config import load_config, resolve_path
from src.demand.data.features import CATEGORICAL_COLS, build_features, feature_columns
from src.demand.data.load import build_panel, load_raw_data
from src.demand.sanity_check.kaggle_pipeline import (
    DEFAULT_LGBM_PARAMS,
    HOLDOUT_END,
    HOLDOUT_START,
    KAGGLE_AS_OF,
    KAGGLE_TEST_END,
    KAGGLE_TEST_START,
    compute_rmsle,
)

cfg = load_config("default.yaml")
results_dir = resolve_path(cfg, "results_dir") / "sanity_check"
results_dir.mkdir(parents=True, exist_ok=True)

FEATURE_SET = "full"
print(f"feature set: {FEATURE_SET}")
print(f"as_of:       {KAGGLE_AS_OF.date()}")
print(f"holdout:     {HOLDOUT_START.date()} .. {HOLDOUT_END.date()}")
print(f"kaggle test: {KAGGLE_TEST_START.date()} .. {KAGGLE_TEST_END.date()}")
print(f"submission → {results_dir / 'submission.csv'}")

## 1. Load raw CSVs

`load_raw_data` reads `train`, `test`, `stores`, `oil`, `holidays_events`, `transactions`, forward-fills oil and verifies the train panel is complete.

In [ ]:
t0 = time.time()
raw = load_raw_data(cfg)
print(f"loaded in {time.time() - t0:.1f}s")
print(f"train rows:        {len(raw.train):>10,}")
print(f"test rows:         {len(raw.test):>10,}")
print(f"stores:            {len(raw.stores):>10,}")
print(f"holidays_events:   {len(raw.holidays):>10,}")
print(f"transactions:      {len(raw.transactions):>10,}")
print(f"oil daily (ffill): {len(raw.oil):>10,}")
raw.train.head(3)

## 2. Build the merged panel including test rows

`build_panel(raw, include_test=True)` returns the long-format `(date, store_nbr, family)` panel with `sales` left as NaN on test-window rows. That NaN gap is what lets `build_features` compute lag features that look back across the train/test boundary without zero-poisoning them.

In [ ]:
panel = build_panel(raw, include_test=True)
print(f"panel rows:  {len(panel):,}")
print(f"date range:  {panel['date'].min().date()} → {panel['date'].max().date()}")
print(f"sales NaN:   {int(panel['sales'].isna().sum()):,} (should equal #test rows = {len(raw.test):,})")
panel.head(3)

## 3. Build engineered features at `as_of = 2017-08-15`

Single feature build for both training and Kaggle test rows. Per PRD §11.3 the sanity-check pipeline uses one cutoff so train/predict feature distributions match — see the comment in `kaggle_pipeline.py` for why we don't use a smaller `as_of` for the holdout.

In [ ]:
t0 = time.time()
features_all = build_features(
    panel,
    feature_set=FEATURE_SET,
    as_of=KAGGLE_AS_OF,
    config=cfg,
    holidays=raw.holidays,
    stores=raw.stores,
)
feat_cols = feature_columns(FEATURE_SET, include_earthquake=True)
print(f"features built in {time.time() - t0:.1f}s")
print(f"frame shape: {features_all.shape}")
print(f"feature cols ({len(feat_cols)}): {feat_cols}")
features_all.head(3)

## 4. Quick sanity glance at feature coverage

Optional. Useful for spotting columns that are unexpectedly all-NaN (sign of a join misalignment) or that have only one unique value (sign of a static-feature bug).

In [ ]:
coverage = pd.DataFrame({
    "non_null_pct": features_all[feat_cols].notna().mean().mul(100).round(1),
    "n_unique":     features_all[feat_cols].nunique(dropna=True),
    "dtype":        features_all[feat_cols].dtypes.astype(str),
}).sort_values("non_null_pct")
coverage

## 5. Local holdout split (last 16 labeled days)

Per PRD §11.3 we hold out 2017-07-31 → 2017-08-15 as a local sanity check (matches the Kaggle test window length). This pass is informative *only* — the real submission uses all labeled data and is trained in step 7.

In [ ]:
train_mask = features_all["date"] < HOLDOUT_START
holdout_mask = (features_all["date"] >= HOLDOUT_START) & (features_all["date"] <= HOLDOUT_END)

def _encode(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    for c in CATEGORICAL_COLS:
        if c in X.columns:
            X[c] = X[c].astype("category")
    return X

X_tr = _encode(features_all.loc[train_mask, feat_cols])
y_tr = features_all.loc[train_mask, "sales"].fillna(0)
X_ho = _encode(features_all.loc[holdout_mask, feat_cols])
y_ho = features_all.loc[holdout_mask, "sales"].fillna(0)

print(f"train rows:   {len(X_tr):>10,}")
print(f"holdout rows: {len(X_ho):>10,}")
print(f"# features:   {X_tr.shape[1]}")

## 6. Train LightGBM (Tweedie) on the train slice and score the holdout

Defaults from `kaggle_pipeline.DEFAULT_LGBM_PARAMS`. Expected RMSLE band per PRD §11.5: **0.45–0.55**.

In [ ]:
cats = [c for c in CATEGORICAL_COLS if c in X_tr.columns]
model = lgb.LGBMRegressor(**DEFAULT_LGBM_PARAMS)
model.fit(X_tr, y_tr, categorical_feature=cats)

y_ho_pred = np.clip(model.predict(X_ho), 0, None)
local_rmsle = compute_rmsle(y_ho, y_ho_pred)
local_mae = float(np.mean(np.abs(y_ho.values - y_ho_pred)))
print(f"local holdout RMSLE: {local_rmsle:.4f}")
print(f"local holdout MAE:   {local_mae:.4f}")
print("PRD red flags:")
print("  • RMSLE < 0.30 → suspect leakage")
print("  • RMSLE > 0.70 → feature/training issue")

## 7. Retrain on all labeled data for the actual Kaggle submission

In [ ]:
full_mask = features_all["date"] <= KAGGLE_AS_OF
X_full = _encode(features_all.loc[full_mask, feat_cols])
y_full = features_all.loc[full_mask, "sales"].fillna(0)
print(f"full train rows: {len(X_full):,}")

model_full = lgb.LGBMRegressor(**DEFAULT_LGBM_PARAMS)
t0 = time.time()
model_full.fit(X_full, y_full, categorical_feature=cats)
print(f"trained in {time.time() - t0:.1f}s")

## 8. Predict the Kaggle test window (2017-08-16 → 2017-08-31)

In [ ]:
test_mask = (features_all["date"] >= KAGGLE_TEST_START) & (features_all["date"] <= KAGGLE_TEST_END)
X_test = _encode(features_all.loc[test_mask, feat_cols])
ids = features_all.loc[test_mask, "id"]
y_test_pred = np.clip(model_full.predict(X_test), 0, None)
print(f"predicted rows: {len(y_test_pred):,} (sample_submission has 28,512)")
pd.Series(y_test_pred).describe().round(2)

## 9. Populate the submission file

Built by aligning predicted ids onto a copy of `sample_submission.csv` so any test row our pipeline misses keeps the sample's default `0.0` rather than silently dropping out of the submission.

In [ ]:
sample_path = resolve_path(cfg, "data_dir") / "sample_submission.csv"
submission = pd.read_csv(sample_path)
preds = pd.DataFrame({"id": ids.values, "sales": y_test_pred})
submission = (
    submission.drop(columns=["sales"])
    .merge(preds, on="id", how="left")
)
missing = submission["sales"].isna().sum()
if missing:
    print(f"WARNING: {missing} ids missing from predictions; filling with 0.0")
    submission["sales"] = submission["sales"].fillna(0.0)
submission["sales"] = submission["sales"].clip(lower=0)
submission_path = results_dir / "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"wrote {submission_path}  ({len(submission):,} rows)")
submission.head(5)

## 10. Persist run metrics

In [ ]:
metrics = {
    "local_holdout_rmsle": local_rmsle,
    "local_holdout_mae": local_mae,
    "n_train_rows": int(len(X_tr)),
    "n_holdout_rows": int(len(X_ho)),
    "n_full_train_rows": int(len(X_full)),
    "n_test_rows": int(len(X_test)),
    "n_features": int(len(feat_cols)),
    "feature_set": FEATURE_SET,
    "timestamp": pd.Timestamp.now("UTC").isoformat(),
}
(results_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
metrics